In [8]:
# Setup para estrutura: .../voidknee/src/{compiler, notebooks, out}
import sys
from pathlib import Path

NB_DIR = Path.cwd()
SRC = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR   # -> .../src
OUT = SRC / "out"
OUT.mkdir(parents=True, exist_ok=True)

# Importa o compilador V3 do pacote 'compiler'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from compiler.voidknee_compiladorV3 import traduzir, ErroCompilacaoVoidKnee

# Helpers para gerar código C e "assembly" didático em src/out
def gerar_c(nome: str, fonte_vk: str, otimizacao: str = "O0") -> str:
    """Compila VoidKnee V3 para C e grava em src/out/<nome>_V3.c"""
    c_code = traduzir(fonte_vk, backend="c", otimizacao=otimizacao)
    destino = OUT / f"{nome}_V3.c"
    destino.write_text(c_code, encoding="utf-8")
    print(f"Gerado C ({otimizacao}) em: {destino.resolve()}")
    return c_code

def gerar_asm(nome: str, fonte_vk: str, otimizacao: str = "O0") -> str:
    """Compila VoidKnee V3 para assembly de máquina de pilha didática"""
    asm_code = traduzir(fonte_vk, backend="asm", otimizacao=otimizacao)
    destino = OUT / f"{nome}_V3.asm"
    destino.write_text(asm_code, encoding="utf-8")
    print(f"Gerado ASM ({otimizacao}) em: {destino.resolve()}")
    return asm_code

def tentar_compilar(descricao: str, fonte_vk: str, **kwargs):
    """Utilitário para mostrar erros didáticos direto no notebook."""
    print(f"### {descricao}")
    try:
        saida = traduzir(fonte_vk, **kwargs)
        print(saida)
    except ErroCompilacaoVoidKnee as e:
        print("ERRO DIDÁTICO:", e)

print("SRC =", SRC.resolve())
print("OUT =", OUT.resolve())


SRC = C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src
OUT = C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out


# VoidKnee V3 🦵 → Tradutor para C **e Assembly didático**

> Evolução da linguagem de brinquedo VoidKnee, agora com **funções**, **recursão**, **casting explícito**,  
> **erros didáticos com linha/coluna**, níveis de **otimização (O0/O1/O2)** e um backend alternativo em **assembly de máquina de pilha**.

---

## O que há de novo na V3 (comparado à V2)

Na V2 nós já tínhamos:

- Léxico, parser, AST e análise semântica para:  
  `inteirao`, `flutuante`, `dobradura`, `verdadeQueDoi`, `dorzinha`, `dorzona`.  
- Controle de fluxo com `sejoelho` / `outracoisa`, `enquantoDoi`, `praCada`.  
- Entrada e saída com `mostraAi(...)` e `entradaAi(...)`.  
- Geração de C com `printf/scanf` e `setlocale` para acentuação.

A V3 mantém esse espírito, mas muda a **arquitetura** e adiciona:

1. **Funções e recursão**  
   - Declarações no estilo C:  
     ```voidknee
     inteirao fatorial(inteirao n) {
         sejoelho (n <= 1) {
             retorna 1;
         } outracoisa {
             retorna n * fatorial(n - 1);
         }
     }
     ```
   - Ponto de entrada explícito: **`inteirao principal()`** (equivalente ao `main` em C).

2. **Casting explícito**  
   - Sintaxe C‑like: `(inteirao) expr`, `(flutuante) expr`, `(dobradura) expr`, `(verdadeQueDoi) expr`.  
   - Útil para controlar promoções de tipo e para corrigir erros de atribuição de forma explícita.

3. **Erros didáticos** (`ErroCompilacaoVoidKnee`)  
   - Toda falha léxica/sintática/semântica traz: **etapa**, **linha** e **coluna**.  
   - Mensagens pensadas para aluno, com dicas (ex.: “use um casting explícito, ex.: `(inteirao) ...`”).

4. **Otimização O0 / O1 / O2**  
   - `O0`: só passa pelo pipeline normal (sem otimização).  
   - `O1`: *constant folding* simples em expressões (`2 + 3 * 4`, `1 && 0`, etc.).  
   - `O2`: herda O1 e ainda faz pequenos *peelings* de código morto, como `sejoelho (0) { ... }`.

5. **Backend alternativo em “assembly”**  
   - Máquina de pilha fictícia, com instruções como `PUSH`, `ADD`, `CALL`, `RET`, `JZ`.  
   - Serve para **visualizar o que o compilador faria antes do C**, mostrando a ligação entre AST e instruções de baixo nível.

---

A ideia da V3 é mais “engenharia de compiladores”: além de traduzir VoidKnee para C,  
mostramos **funções reais, recursion, otimização e múltiplos backends** em um código didático e enxuto.


## Recapitulação rápida do vocabulário VoidKnee (versão V3)

### Tipos básicos

| VoidKnee        | Significado            | Mapeamento C  | Observações |
|-----------------|------------------------|---------------|------------|
| `inteirao`      | inteiro                | `int`         | tipo padrão para contadores e retornos simples |
| `flutuante`     | ponto flutuante        | `float`       | |
| `dobradura`     | dupla precisão         | `double`      | |
| `verdadeQueDoi` | booleano               | `int` (0 / 1) | literais: `verdadeiro`, `falso` |
| `dorzinha`      | caractere              | `char`        | |
| `dorzona`       | string                 | `char*`       | na V3 usamos ponteiro simples em C (sem arrays) |

### Controle de fluxo e funções

- `sejoelho (cond) { ... } outracoisa { ... }`
- `enquantoDoi (cond) { ... }`
- `praCada (inicial; cond; passo) { ... }`
- Declaração de função:
  ```voidknee
  tipo nome(tipo p1, tipo p2, ...) {
      // corpo
      retorna valor;
  }
  ```
- Ponto de entrada:
  ```voidknee
  inteirao principal() {
      // ...
      retorna 0;
  }
  ```

### Entrada / saída

- `mostraAi(expr1, expr2, ...);` — mapeado para `printf`, com tratamento simplificado na V3.  
- `entradaAi(x);` — lê para uma variável simples (por enquanto focado em `inteirao`).

### Casting e operadores

- **Casting explícito**: `(inteirao) expr`, `(flutuante) expr`, `(dobradura) expr`, `(verdadeQueDoi) expr`.  
  Usado para ajustar tipos em expressões e atribuições.
- Aritméticos: `+`, `-`, `*`, `/`
- Relacionais: `<`, `<=`, `>`, `>=`, `==`, `!=`
- Lógicos: `&&`, `||`, `!`

> Nota: vários exemplos da V2 (vetores/matrizes, strings avançadas) continuam válidos como **ideia de linguagem**,  
> mas na V3 o foco do compilador está em **funções, recursão, casting, otimização e assembly**, então os exemplos aqui vão nessa direção.


# Explicação **passo a passo** da V3

> Agora o pipeline vai além do “tokenizar → AST → C” e passa por **funções**, **escopos**,  
> **casting**, **otimização** e escolha de **backend** (`c` ou `asm`).

---

## 1) `traduzir` — orquestração com backend e otimização

A função de alto nível da V3 é:

```python
def traduzir(fonte: str, *, backend: str = "c", otimizacao: str = "O0") -> str:
    lexer = Lexer(fonte)
    tokens = lexer.tokens()
    parser = Parser(tokens)
    prog = parser.parse()

    analise = AnalisadorSemantico(prog)
    analise.analisar()

    prog = otimizar_programa(prog, otimizacao)

    if backend == "asm":
        return GeradorAssembly(prog).gerar()
    return GeradorC(prog).gerar()
```

Resumo:

1. **Lexer**: converte texto em tokens (`IDENT`, `INT`, `TIPO`, `KW_SE`, operadores, etc.).  
2. **Parser**: monta a AST (`Programa` com **funções**, variáveis globais e blocos).  
3. **Analisador semântico**: verifica tipos, escopos e chamadas de função.  
4. **Otimizador**: aplica *constant folding* e simplificações leves (dependendo de `O0/O1/O2`).  
5. **Backend**: gera **C** ou **assembly de pilha**.

---

## 2) Léxico — `Lexer.tokens()`

O léxico da V3:

- Reconhece **palavras‑chave** (`sejoelho`, `outracoisa`, `enquantoDoi`, `praCada`, `retorna`, `verdadeiro`, `falso`).  
- Reconhece **tipos VoidKnee** (`inteirao`, `flutuante`, `dobradura`, `dorzinha`, `dorzona`, `verdadeQueDoi`).  
- Emite `TokenTipo.OP` para operadores (`+`, `-`, `*`, `/`, `==`, `!=`, `&&`, `||`, ...).  
- Emite `TokenTipo.PONT` para pontuação (`(`, `)`, `{`, `}`, `,`, `;`, `[]`).  
- Ignora comentários de linha (`// ...`) e de bloco (`/* ... */`).  
- Em caso de erro (string não fechada, comentário não fechado, caractere desconhecido),  
  lança `ErroCompilacaoVoidKnee` com etapa `"léxico"`.

---

## 3) AST — funções, escopos e casting

Alguns nós importantes da AST:

```python
@dataclass
class FuncDecl:
    tipo_retorno: Tipo
    nome: str
    params: List[Param]
    corpo: Block

@dataclass
class Call(Expr):
    nome: str
    argumentos: List[Expr]

@dataclass
class Cast(Expr):
    tipo_destino: Tipo
    expr: Expr
```

- O programa (`Programa`) agora tem **lista de funções** e **globais**.  
- `FuncDecl` representa uma função com tipo de retorno, parâmetros e um `Block` de comandos.  
- `Call` e `Cast` permitem modelar **chamadas de função** e **casting explícito** na árvore.

---

## 4) Parser — funções, `principal` e casting estilo C

O parser reconhece duas formas elementares no topo do arquivo:

- **Declaração global**: `inteirao x;`, `flutuante pi = 3.14;`  
- **Declaração de função**: `inteirao f(inteirao n) { ... }`

A distinção é feita ao ver se, após o identificador, vem um `(` (função) ou não (variável global).

Para o **casting**, usamos um *lookahead* parecido com C:

```python
def _cast(self) -> Expr:
    if self._atual().tipo == TokenTipo.PONT and self._atual().lexema == "(":
        salva_i = self.i
        self._consumir(TokenTipo.PONT, "(")
        if self._atual().tipo == TokenTipo.TIPO:
            tipo_tok = self._consumir(TokenTipo.TIPO)
            if self._atual().tipo == TokenTipo.PONT and self._atual().lexema == ")":
                self._consumir(TokenTipo.PONT, ")")
                expr = self._cast()
                return Cast(TIPO_POR_NOME[tipo_tok.lexema], expr, tipo_tok.linha, tipo_tok.coluna)
        self.i = salva_i  # não era cast, volta e trata como expressão normal
    return self._primario()
```

Isso permite escrever:

```voidknee
inteirao x;
flutuante y;
x = (inteirao) y + 1;
```

---

## 5) Semântica — escopos aninhados, funções e erros didáticos

O analisador semântico constrói:

- Uma tabela de **funções** (`self.funcoes: Dict[str, AssinaturaFunc]`).  
- Uma hierarquia de **escopos** (`Escopo`) para variáveis globais, parâmetros e locais.

Exemplo de checagem de chamada de função:

```python
if expr.nome not in self.funcoes:
    raise ErroCompilacaoVoidKnee(
        f"Chamada a função desconhecida '{expr.nome}'.",
        expr.linha, expr.coluna, "semântico",
    )
ass = self.funcoes[expr.nome]
if len(expr.argumentos) != len(ass.params):
    raise ErroCompilacaoVoidKnee(
        f"Função '{expr.nome}' espera {len(ass.params)} argumentos, mas recebeu {len(expr.argumentos)}.",
        expr.linha, expr.coluna, "semântico",
    )
```

A verificação de atribuição também é didática:

```python
def _verifica_atribuicao(self, destino: Tipo, fonte: Tipo, linha: int, coluna: int):
    if destino.prim == fonte.prim:
        return
    if destino.prim in (INTEIRO, FLUTUANTE, DOBRADURA) and fonte.prim in (...):
        return
    raise ErroCompilacaoVoidKnee(
        f"Não é possível atribuir valor do tipo {fonte.prim.name} em variável do tipo {destino.prim.name}. "
        f"Dica: use um casting explícito, ex.: (inteirao) ...",
        linha, coluna, "semântico",
    )
```

Resultado: se você tentar fazer `inteirao x; dorzona s; x = s;` o compilador acusa o tipo e ainda sugere casting.

Também é nesta fase que exigimos a existência de `inteirao principal()` como ponto de entrada.

---

## 6) Otimização — O0, O1 e O2

A função `otimizar_programa` aplica transformações leves na AST:

- `O0`: nenhuma otimização.  
- `O1`: *constant folding* em expressões (`2 + 3 * 4`, `1 && 0`, `(inteirao) 3.7` etc.).  
- `O2`: além de O1, remove alguns statements redundantes, por exemplo:
  - `sejoelho (0) { ... }` é eliminado.  
  - `sejoelho (1) { blocoA } outracoisa { blocoB }` vira só `blocoA`.  
  - `ExprStmt` que virou apenas literal é descartado.

Tudo isso é feito ainda na AST, antes da geração de C/ASM, para facilitar a visualização.

---

## 7) Backends — C e Assembly de pilha

### Backend C (`GeradorC`)

- Declara variáveis globais no topo.  
- Emite protótipos e, em seguida, todas as funções da linguagem.  
- Se existir `inteirao principal()`, gera automaticamente:

  ```c
  int main(void) {
      return principal();
  }
  ```

- Dentro de `principal`, o compilador insere `setlocale(LC_ALL, "")` para lidar com acentos no Windows.

### Backend Assembly (`GeradorAssembly`)

- Gera rótulos por função: `fatorial:`, `principal:` etc.  
- Usa instruções simples de pilha: `PUSH`, `POP`, `LOAD`, `STORE`, `ADD`, `SUB`, `MUL`, `DIV`, `CALL`, `RET`, `JZ`, `JMP`.  
- Estruturas como `sejoelho` e `enquantoDoi` viram combinações de `JZ`/`JMP` com rótulos (`else_1`, `endwhile_2`, ...).

Isso tudo torna a V3 um ótimo material para **apresentar em sala**: dá para mostrar o mesmo programa em nível de
linguagem VoidKnee, C gerado e assembly didático.


## Exemplos (gerando C / ASM)


### A) Função recursiva — `fatorial`

Demonstra:

- Declaração de função com parâmetro.  
- Recursão.  
- Uso de `retorna` dentro de `sejoelho`.  
- Ponto de entrada `inteirao principal()` chamando a função.


In [9]:
# Exemplo A — fatorial recursivo

source = """
inteirao fatorial(inteirao n) {
    sejoelho (n <= 1) {
        retorna 1;
    } outracoisa {
        retorna n * fatorial(n - 1);
    }
}

inteirao principal() {
    inteirao x;
    x = 5;
    mostraAi("Fatorial de 5 = ");
    mostraAi(x);  // V3 usa printf simplificado
    retorna fatorial(x);
}
"""

c_code = gerar_c("v3_A_fatorial", source, otimizacao="O0")
print(c_code)


Gerado C (O0) em: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v3_A_fatorial_V3.c
#include <stdio.h>
#include <locale.h>

/* Funções auxiliares para VoidKnee V3 */

int fatorial(int n);
int principal();

int fatorial(int n) {
    if ((n <= 1)) {
        return 1;
    }
    else {
        return (n * fatorial((n - 1)));
    }
    return 0;
}

int principal() {
    setlocale(LC_ALL, "");
    int x;
    x = 5;
    printf("%s", "Fatorial de 5 = ");
    printf("[mostraAi simplificado] %d\n", x);
    return fatorial(x);
}

int main(void) {
    return principal();
}



### B) Funções que se chamam e retorno de valor

Exemplo com duas funções:

- `inteirao quadrado(inteirao x)`  
- `inteirao soma_quadrados(inteirao a, inteirao b)`  
- `principal` faz a chamada e retorna o resultado.


In [20]:
# Exemplo B — múltiplas funções e chamadas encadeadas

source = """
inteirao quadrado(inteirao x) {
    retorna x * x;
}

inteirao soma_quadrados(inteirao a, inteirao b) {
    inteirao sa;
    sa = quadrado(a) + quadrado(b);
    retorna sa;
}

inteirao principal() {
    inteirao r;
    r = soma_quadrados(3, 4);
    mostraAi("Soma dos quadrados de 3 e 4 = ");
    mostraAi(r);
    retorna r;
}
"""

c_code = gerar_c("v3_B_funcoes", source, otimizacao="O0")
print(c_code)


Gerado C (O0) em: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v3_B_funcoes_V3.c
#include <stdio.h>
#include <locale.h>

/* Funções auxiliares para VoidKnee V3 */

int quadrado(int x);
int soma_quadrados(int a, int b);
int principal();

int quadrado(int x) {
    return (x * x);
}

int soma_quadrados(int a, int b) {
    int sa;
    sa = (quadrado(a) + quadrado(b));
    return sa;
}

int principal() {
    setlocale(LC_ALL, "");
    int r;
    r = soma_quadrados(3, 4);
    printf("%s", "Soma dos quadrados de 3 e 4 = ");
    printf("[mostraAi simplificado] %d\n", r);
    return r;
}

int main(void) {
    return principal();
}



### C) Casting explícito e erros didáticos

Aqui mostramos:

- Como usar `(flutuante)` e `(inteirao)` para controlar o tipo.  
- Um exemplo em que o compilador recusa a atribuição e sugere usar casting.


In [19]:
# Exemplo C1 — casting numérico válido

source_ok = """
inteirao principal() {
    inteirao x;
    flutuante y;
    dobradura z;

    y = 3.5;
    z = (dobradura) y * 2;
    x = (inteirao) z;  // truncando de forma explícita

    retorna x;
}
"""

c_code_ok = gerar_c("v3_C_casting_ok", source_ok, otimizacao="O1")
print(c_code_ok)

# Exemplo C2 — erro didático de tipos (string → inteiro)
source_erro = """
inteirao principal() {
    inteirao x;
    dorzona texto;

    x = texto;  // aqui esperamos que o compilador reclame
    retorna 0;
}
"""

tentar_compilar("Casting inválido (dorzona → inteirao)", source_erro, backend="c", otimizacao="O0")


Gerado C (O1) em: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v3_C_casting_ok_V3.c
#include <stdio.h>
#include <locale.h>

/* Funções auxiliares para VoidKnee V3 */

int principal();

int principal() {
    setlocale(LC_ALL, "");
    int x;
    float y;
    double z;
    y = 3.5f;
    z = (((double) y) * 2);
    x = ((int) z);
    return x;
}

int main(void) {
    return principal();
}

### Casting inválido (dorzona → inteirao)
ERRO DIDÁTICO: [semântico erro] linha 6, coluna 7: Não é possível atribuir valor do tipo STRING em variável do tipo INTEIRO. Dica: use um casting explícito, ex.: (inteirao) ...


### D) Comparando O0, O1 e O2

Vamos usar um programa com várias expressões constantes para visualizar:

- *Constant folding* (`2 + 3 * 4`, `1 && 0`, etc.).  
- Remoção de `sejoelho (0) { ... }` em O2.


In [18]:
# Exemplo D — diferenças entre O0, O1 e O2

source = """
inteirao principal() {
    inteirao x;
    x = 2 + 3 * 4;          // 14

    sejoelho (1 && 0) {
        mostraAi("Nunca deveria aparecer\n");
    }

    sejoelho (2 * 3 > 5) {
        mostraAi("Essa condição é sempre verdadeira\n");
    }

    retorna x;
}
"""

for nivel in ["O0", "O1", "O2"]:
    print("\n==========================")
    print("Nível de otimização:", nivel)
    print("==========================")
    c_code = gerar_c(f"v3_D_otimizacao_{nivel}", source, otimizacao=nivel)
    print(c_code)



Nível de otimização: O0
Gerado C (O0) em: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v3_D_otimizacao_O0_V3.c
#include <stdio.h>
#include <locale.h>

/* Funções auxiliares para VoidKnee V3 */

int principal();

int principal() {
    setlocale(LC_ALL, "");
    int x;
    x = (2 + (3 * 4));
    if ((1 && 0)) {
        printf("%s", "Nunca deveria aparecer
");
    }
    if (((2 * 3) > 5)) {
        printf("%s", "Essa condição é sempre verdadeira
");
    }
    return x;
}

int main(void) {
    return principal();
}


Nível de otimização: O1
Gerado C (O1) em: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v3_D_otimizacao_O1_V3.c
#include <stdio.h>
#include <locale.h>

/* Funções auxiliares para VoidKnee V3 */

int principal();

int principal() {
    setlocale(LC_ALL, "");
    int x;
    x = 14;
    if (0) {
        printf("%s", "Nunca deveria aparecer
");
    }
    if (1) {
        printf("%s", "Essa condição é s

### E) Backend de assembly de pilha

Além do C, a V3 consegue gerar um **assembly de máquina de pilha didática**, usando `backend="asm"`.
Isso permite enxergar:

- Como expressões viram sequência de instruções (`PUSH`, `ADD`, `MUL`, …);  
- Como `sejoelho` / `enquantoDoi` viram rótulos + saltos (`JZ`, `JMP`);  
- Como chamadas de função são mapeadas para `CALL` / `RET`.

O assembly gerado é conceitual, então criamos também uma **VM em Python** (`vm_voidknee.py`) que interpreta esse código.

#### Executando o assembly gerado

1. Gerar o `.asm` pelo compilador (por exemplo, `v3_E_fatorial_asm_V3.asm` em `src/out`);  
2. Rodar a VM apontando para o arquivo e para o rótulo de entrada:

```bash
py src/vm_voidknee.py src/out/v3_E_fatorial_asm_V3.asm principal


In [17]:
# Exemplo E — gerando assembly para o fatorial

source = """
inteirao fatorial(inteirao n) {
    sejoelho (n <= 1) {
        retorna 1;
    } outracoisa {
        retorna n * fatorial(n - 1);
    }
}

inteirao principal() {
    inteirao x;
    x = 5;
    retorna fatorial(x);
}
"""

asm_code = gerar_asm("v3_E_fatorial_asm", source, otimizacao="O1")
print(asm_code)


Gerado ASM (O1) em: C:\Users\user\Documents\Puc\Compiladores\desenvolvimento compilador\voidknee\src\out\v3_E_fatorial_asm_V3.asm
; Assembly VoidKnee V3 - máquina de pilha didática
; Este backend é conceitual e não é montado por um assembler real.

fatorial:
    ; prólogo (simples)
    LOAD n
    PUSH 1
    LE
    JZ else_7
    PUSH 1
    RET
    JMP endif_8
else_7:
    LOAD n
    LOAD n
    PUSH 1
    SUB
    CALL fatorial
    MUL
    RET
endif_8:
    PUSH 0 ; retorno padrão caso não haja 'retorna'
    RET

principal:
    ; prólogo (simples)
    ; var x : INTEIRO
    PUSH 5
    STORE x
    LOAD x
    POP
    LOAD x
    CALL fatorial
    RET
    PUSH 0 ; retorno padrão caso não haja 'retorna'
    RET



### F) Mais um exemplo de erro didático — chamada de função incorreta

Exemplo de dois erros comuns:

- Chamar função que não existe.  
- Passar número errado de argumentos.


In [15]:
# Exemplo F — erros de função

fonte_fun_desconhecida = """
inteirao principal() {
    inteirao x;
    x = funcaoInexistente(10);
    retorna x;
}
"""

tentar_compilar("Função desconhecida", fonte_fun_desconhecida, backend="c", otimizacao="O0")

fonte_args_errados = """
inteirao soma(inteirao a, inteirao b) {
    retorna a + b;
}

inteirao principal() {
    inteirao r;
    r = soma(10);   // faltou um argumento
    retorna r;
}
"""

tentar_compilar("Número errado de argumentos", fonte_args_errados, backend="c", otimizacao="O0")


### Função desconhecida
ERRO DIDÁTICO: [semântico erro] linha 4, coluna 9: Chamada a função desconhecida 'funcaoInexistente'.
### Número errado de argumentos
ERRO DIDÁTICO: [semântico erro] linha 8, coluna 9: Função 'soma' espera 2 argumentos, mas recebeu 1.


## 👏 Sessão de Palmas da V3

> *aplausos ecoam pela sala* 👏👏👏👏👏  
> Compilador agora com **funções, recursão, casting, otimização e assembly**. 🧠⚙️

- Mostramos como a VoidKnee V3 evolui a partir da V2.  
- Criamos programas com funções recursivas e vimos o C/ASM gerados.  
- Exploramos erros didáticos, que ajudam a entender melhor tipos e escopos.  
- Visualizamos otimizações simples (O0/O1/O2) direto no código gerado.

**Obrigado! equipe VoidKnee V3!** 🦵🚀
